# Practical 6 — Object Detection using Transfer Learning of a Pre-trained CNN

**Aim:** Perform object detection using a pre-trained CNN architecture and transfer learning.

**Implementation note:** This notebook uses TorchVision's pre-trained Faster R-CNN model (COCO weights). Its backbone has already learned visual features from a large dataset. We run inference on an image and draw predicted bounding boxes. This demonstrates use of a pre-trained detection architecture; it does not fine-tune a custom dataset because that requires labelled bounding-box training data.

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

## Load a pre-trained detector

The first run downloads the model weights, so internet access is required unless weights are already cached. We use the default COCO-pretrained weights and their class labels.

In [ ]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()
categories = weights.meta["categories"]
preprocess = weights.transforms()
print("Pre-trained Faster R-CNN loaded.")

## Load an image

Place an image named `sample.jpg` in the `dataset` folder, or enter the path to any image file available on the computer.

In [ ]:
image_path = Path("dataset/sample.jpg")
if not image_path.exists():
    image_path = Path("../dataset/sample.jpg")

if not image_path.exists():
    raise FileNotFoundError(
        "Put an image named sample.jpg in the dataset folder, then rerun this cell."
    )

image = Image.open(image_path).convert("RGB")
image_tensor = preprocess(image)
print("Image size:", image.size)

In [ ]:
with torch.no_grad():
    prediction = model([image_tensor])[0]

threshold = 0.5
fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(image)

for box, label, score in zip(
    prediction["boxes"], prediction["labels"], prediction["scores"]
):
    if score >= threshold:
        x1, y1, x2, y2 = box.tolist()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1, f"{categories[int(label)]}: {float(score):.2f}",
            color="white", backgroundcolor="red"
        )

ax.axis("off")
plt.show()

## Transfer learning concepts / viva
1. What is transfer learning? — Reusing knowledge learned by a model on a source task for a related target task.
2. What is a pre-trained model? — A model whose weights were learned previously on a dataset.
3. What is a CNN backbone? — The feature-extraction part of a CNN.
4. What does freezing layers mean? — Preventing their weights from being updated during training.
5. What is fine-tuning? — Unfreezing some pre-trained layers and training them further on target data.
6. What is object detection? — Identifying object classes and locating them with bounding boxes.
7. What is confidence score? — The model's confidence in a predicted detection.

**Important distinction:** This notebook performs inference with a pre-trained detector. The manual's full transfer-learning training workflow (freeze lower layers, add/train a custom classifier, then fine-tune) needs a labelled target dataset and training code specific to that dataset.

**Conclusion:** Loaded a pre-trained Faster R-CNN model and detected objects with bounding boxes.